# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 1: Data Collection via SpaceX REST API
**Author: Gautam825406**

In [ ]:
# Install required libraries
!pip install requests pandas numpy --quiet

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime

# SpaceX API base URL
spacex_url = 'https://api.spacexdata.com/v4/launches/past'

print('Libraries imported successfully')

## Task 1: Request SpaceX launches data

In [ ]:
response = requests.get(spacex_url)
print('Status Code:', response.status_code)
data = response.json()
print('Total launches:', len(data))

In [ ]:
# Helper functions to get additional info from API
def getBoosterVersion(data):
    BoosterVersion = []
    for x in data['rocket']:
        if x:
            response = requests.get('https://api.spacexdata.com/v4/rockets/'+str(x)).json()
            BoosterVersion.append(response['name'])
        else:
            BoosterVersion.append(None)
    return BoosterVersion

def getLaunchSite(data):
    Longitude = []
    Latitude = []
    LaunchSite = []
    for x in data['launchpad']:
        if x:
            response = requests.get('https://api.spacexdata.com/v4/launchpads/'+str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])
        else:
            Longitude.append(None)
            Latitude.append(None)
            LaunchSite.append(None)
    return Longitude, Latitude, LaunchSite

def getPayloadData(data):
    PayloadMass = []
    Orbit = []
    for load in data['payloads']:
        if load:
            response = requests.get('https://api.spacexdata.com/v4/payloads/'+load).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])
        else:
            PayloadMass.append(None)
            Orbit.append(None)
    return PayloadMass, Orbit

def getCoreData(data):
    Block = []
    ReusedCount = []
    Serial = []
    Outcome = []
    Flights = []
    GridFins = []
    Reused = []
    Legs = []
    LandingPad = []
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get('https://api.spacexdata.com/v4/cores/'+core['core']).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])
    return Block, ReusedCount, Serial, Outcome, Flights, GridFins, Reused, Legs, LandingPad

print('Helper functions defined')

## Task 2: Filter Falcon 9 launches and build DataFrame

In [ ]:
# Use static dataset from IBM for consistency
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv')
print('Dataset shape:', df.shape)
df.head(10)

In [ ]:
# Check data types
df.dtypes

In [ ]:
# Check for missing values
df.isnull().sum()

## Task 3: Exploratory Analysis - Launch Sites

In [ ]:
# Number of launches at each site
df['LaunchSite'].value_counts()

In [ ]:
# Number of launches per orbit
df['Orbit'].value_counts()

In [ ]:
# Landing outcomes
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes

In [ ]:
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)

In [ ]:
bad_outcomes = set(landing_outcomes.keys()[[1,3,5,6,7]])
print('Bad outcomes:', bad_outcomes)

## Task 4: Create Training Labels

In [ ]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]
df['Class'] = landing_class
df[['Class']].head(8)

In [ ]:
print('Success rate:', df['Class'].mean())
df.head(5)

In [ ]:
df.to_csv('dataset_part_1.csv', index=False)
print('Saved dataset_part_1.csv')